In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json, lit
from pyspark.sql.types import StructType

In [0]:
%run ../../helper_function/helper_function


In [0]:
dbutils.widgets.text("sport", "football", "sport")
sport = dbutils.widgets.get("sport")

In [0]:
# Read control table
control_df = spark.sql(
    f"""
        SELECT 
            *
        FROM  
            workspace.bk_curated.curated_control_tbl
        WHERE 
            sport = '{sport}' 
    """)
control_df.display()

In [0]:
control_rows = control_df.collect()

for row in control_rows:

    team = row["team"]
    tournament = row["tournament"]
    source_uc = row["source_uc"]
    source_uc_schema = row["source_uc_schema"]
    source_uc_table = row["source_uc_table"]
    target_uc = row["target_uc"]
    target_uc_schema = row["target_uc_schema"]
    target_uc_table = row["target_uc_table"]
    json_col = row["json_col"]
    list_json_col = row["list_json_col"]
    final_col = row["final_col"]
    filter_con = row["filter_con"]
    load_type = row["load_type"]
    delta_col = row["delta_col"]
    partition_col = row["partition_col"]

    print(target_uc_table)
    
    raw_df = spark.sql(
        f"SELECT * FROM {source_uc}.{source_uc_schema}.{source_uc_table}"
    )

    if json_col is not None:
        expanded_df =  expand_json_columns(raw_df)
    else:
        expanded_df = raw_df
    
    if list_json_col is not None:
        currated_df =  expand_list_json_columns(expanded_df,list_json_col)
    else:
        currated_df = expanded_df

    # Apply the required final column order
    base_columns = [
        c.strip() for c in final_col.split(",") if c.strip()
    ]

    new_base_columns = []

    for bc in base_columns:
        if "*" in bc:
            new_bc = bc.replace("*","")
            for cu_col in currated_df.columns:
                if new_bc in cu_col:
                    new_base_columns.append(cu_col)
        else:
            new_base_columns.append(bc)
    final_df = currated_df.select(
        *new_base_columns
    )
    
    if filter_con is not None:
        final_df = final_df.filter(filter_con)
    
    final_df = final_df.withColumn('rec_captured_date', F.current_date())

    final_df\
    .write.mode(load_type)\
    .partitionBy(partition_col)\
    .saveAsTable(f"{target_uc}.{target_uc_schema}.{target_uc_table}")
    